In [45]:
import pandas as pd
import statsmodels.api as sm

In [11]:
ru_alphabet = 'абвгдеёжзийклмнопрстуфхцчшщъыьэюя'

In [12]:
first_letter_bobrov = 'б'
first_letter_chunikhina = 'ч'

index_bobrov = ru_alphabet.index(first_letter_bobrov) + 1
index_chunikhina = ru_alphabet.index(first_letter_chunikhina) + 1

(index_bobrov + index_chunikhina) % 4

3

# Бинго! Регрессия Пуассона

Регрессия Пуассона очень требовательна, она требует целевую переменную как переменную количественную (подсчет, целая неотрицательная) и чтобы она находилась примерно в распределении Пуассона. Главный признак распределения Пуассона – среднее значение и дисперсия равны. Постараемся найти такой датасет (это была самая сложная часть работы)

In [33]:
df = pd.read_csv("train.csv")

In [39]:
X, y = df.drop(columns=["Rings", "Sex", "id"]), df["Rings"]

In [40]:
y.describe()

count    90615.000000
mean         9.696794
std          3.176221
min          1.000000
25%          8.000000
50%          9.000000
75%         11.000000
max         29.000000
Name: Rings, dtype: float64

Видим, что стандартное отклонение и среднее у целевой переменной примерно равны и она является количественной, поэтому можем попробовать этот датасет для регрессии Пуассона.

Для интереса сначала посмотрим, как с этими данными справляется обычная линейная регрессия. В качестве метрики выберем RMSE

In [46]:
variables = sm.add_constant(X)

In [47]:
model = sm.Poisson(y, variables)

In [48]:
results = model.fit()

Optimization terminated successfully.
         Current function value: 2.223773
         Iterations 5


In [49]:
print(results.summary())

                          Poisson Regression Results                          
Dep. Variable:                  Rings   No. Observations:                90615
Model:                        Poisson   Df Residuals:                    90607
Method:                           MLE   Df Model:                            7
Date:                Thu, 02 Apr 2026   Pseudo R-squ.:                  0.1203
Time:                        16:27:02   Log-Likelihood:            -2.0151e+05
converged:                       True   LL-Null:                   -2.2906e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const              1.2911      0.010    131.112      0.000       1.272       1.310
Length             0.1798      0.063      2.846      0.004       0.056       0.304
Diameter           1.5093      0.077

## Интерпретация результатов пуассоновской регрессии (Abalone dataset)

### Общая информация о модели
Была построена модель пуассоновской регрессии для объяснения количества колец моллюска

Основные показатели модели:

- Число наблюдений: 90 615
- Pseudo R²: 0.1203
- Log-Likelihood: -2.015×10⁵
- LLR p-value: < 0.001

#### Вывод о значимости модели
Тест отношения правдоподобия (LLR) показывает p-value < 0.001, следовательно, модель статистически значимо лучше нулевой модели без предикторов

Значение Pseudo R² ≈ 0.12 указывает на умеренную объясняющую способность, что является нормальным результатом для биологии

## Интерпретация коэффициентов
В пуассоновской регрессии коэффициенты находятся в логарифмической шкале
Поэтому интерпретация проводится через показатель:

$$
IRR = e^{\beta}
$$

IRR показывает, во сколько раз меняется ожидаемое число колец при увеличении признака на 1 единицу

### Свободный член (Intercept)
- coef = 1.291
- IRR = $e^{1.291}$ = 3.64

Если все предикторы равны нулю, ожидаемое число колец составляет примерно 3.6

## Размерные характеристики моллюска

### Length
- coef = 0.1798 → IRR = 1.20

При увеличении длины на 1 единицу ожидаемое число колец увеличивается на 19.7%.  
Переменная статистически значима (p = 0.004)

### Diameter
- coef = 1.5093 → IRR = 4.52

Увеличение диаметра на 1 единицу связано с увеличением ожидаемого числа колец в 4.5 раза
Это один из сильнейших предикторов возраста

### Height
- coef = 1.9748 → IRR = 7.20

Увеличение высоты на 1 единицу приводит к увеличению числа колец более чем в 7 раз
Это наиболее сильный фактор в модели

## Весовые характеристики

### Whole weight
- coef = 0.2546 → IRR = 1.29

Увеличение общего веса на 1 единицу повышает ожидаемое число колец на ≈29%

### Whole weight.1
- coef = −1.4451 → IRR = 0.24

Увеличение веса мяса связано со снижением ожидаемого числа колец на 76%

### Whole weight.2
- coef = −0.5063 → IRR ≈ 0.60

Увеличение веса внутренностей связано со снижением числа колец примерно на 40%

### Shell weight
- coef = 1.5936 → IRR ≈ 4.92

Увеличение веса раковины на 1 единицу приводит к увеличению числа колец почти в 5 раз

Это один из наиболее логичных и сильных предикторов возраста

## Статистическая значимость предикторов
Все переменные имеют p-value < 0.001, следовательно, каждый предиктор статистически значим

## Важное замечание о мультиколлинеарности
Различные показатели веса сильно коррелируют между собой
Это может приводить к появлению отрицательных коэффициентов у части весовых переменных. Такое поведение типично для набора данных Abalone и не противоречит общей логике модели

## Итоговый вывод
Построенная пуассоновская модель показывает, что:

- Основными факторами возраста моллюска являются высота, диаметр и вес раковины
- Размерные характеристики оказывают наиболее сильное положительное влияние на число колец
- Модель статистически значима и демонстрирует умеренную объясняющую способность, что является ожидаемым результатом для биологических счётных данных